## Install Conda

In [1]:
!pip install -q condacolab
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


In [2]:
!conda -V

conda 24.11.2


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
%cd /content/drive/MyDrive/AbDev

/content/drive/MyDrive/AbDev


## Import input data

In [5]:
import pandas as pd
dataset = pd.read_csv('Sequence_Info.csv') ## read your csv file

In [6]:
dataset

,Name,VH,VL
0,abituzumab,QVQLQQSGGELAKPGASVKVSCKASGYTFSSFWMHWVRQAPGQGLE...,DIQMTQSPSSLSASVGDRVTITCRASQDISNYLAWYQQKPGKAPKL...
1,abrilumab,QVQLVQSGAEVKKPGASVKVSCKVSGYTLSDLSIHWVRQAPGKGLE...,DIQMTQSPSSVSASVGDRVTITCRASQGISSWLAWYQQKPGKAPKL...
2,adalimumab,EVQLVESGGGLVQPGRSLRLSCAASGFTFDDYAMHWVRQAPGKGLE...,DIQMTQSPSSLSASVGDRVTITCRASQGIRNYLAWYQQKPGKAPKL...
3,alemtuzumab,QVQLQESGPGLVRPSQTLSLTCTVSGFTFTDFYMNWVRQPPGRGLE...,DIQMTQSPSSLSASVGDRVTITCKASQNIDKYLNWYQQKPGKAPKL...
4,alirocumab,EVQLVESGGGLVQPGGSLRLSCAASGFTFNNYAMNWVRQAPGKGLD...,DIVMTQSPDSLAVSLGERATINCKSSQSVLYRSNNRNFLGWYQQKP...
...,...,...,...
132,vedolizumab,QVQLVQSGAEVKKPGASVKVSCKGSGYTFTSYWMHWVRQAPGQRLE...,DVVMTQSPLSLPVTPGEPASISCRSSQSLAKSYGNTYLSWYLQKPG...
133,veltuzumab,QVQLQQSGAEVKKPGSSVKVSCKASGYTFTSYNMHWVKQAPGQGLE...,DIQLTQSPSSLSASVGDRVTMTCRASSSVSYIHWFQQKPGKAPKPW...
134,visilizumab,QVQLVQSGAEVKKPGASVKVSCKASGYTFISYTMHWVRQAPGQGLE...,DIQMTQSPSSLSASVGDRVTITCSASSSVSYMNWYQQKPGKAPKRL...
135,zalutumumab,QVQLVESGGGVVQPGRSLRLSCAASGFTFSTYGMHWVRQAPGKGLE...,AIQLTQSPSSLSASVGDRVTITCRASQDISSALVWYQQKPGKAPKL...


In [17]:
# Import libraries
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt

from numpy.random import seed

# Import machine learning libraries
import tensorflow as tf

import keras
from keras.callbacks import ModelCheckpoint
from keras.optimizers import Adam
from keras.models import model_from_json, Sequential
from keras.layers import Conv1D, BatchNormalization, Dropout, MaxPooling1D, Flatten, Dense
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### Generate a FASTA file

In [18]:
import sys, site
print("Python:", sys.version)
print("Executable:", sys.executable)
print("site-packages:", site.getsitepackages())

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Executable: /usr/bin/python3.real
site-packages: ['/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/lib/python3.12/dist-packages']


In [23]:
import sys
!{sys.executable} -m pip install -U biopython anarcii

  Using cached anarcii-2.0.5-py3-none-any.whl.metadata (2.3 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
Using cached anarcii-2.0.5-py3-none-any.whl (17.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 27.8 MB/s eta 0:00:00
Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl (29 kB)


In [24]:
name = dataset['Name'].to_list()
Heavy_seq = dataset['VH'].to_list()
Light_seq = dataset['VL'].to_list()

In [25]:
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

file_out='seq_H.fasta'

with open(file_out, "w") as output_handle:
  for i in range(len(name)):
    seq_name = name[i]
    seq = Heavy_seq[i]
    record = SeqRecord(
    Seq(seq),
    id=seq_name,
    name="",
    description="",
    )
    SeqIO.write(record, output_handle, "fasta")

file_out='seq_L.fasta'

with open(file_out, "w") as output_handle:
  for i in range(len(name)):
    seq_name = name[i]
    seq = Light_seq[i]
    record = SeqRecord(
    Seq(seq),
    id=seq_name,
    name="",
    description="",
    )
    SeqIO.write(record, output_handle, "fasta")

### Sequence alignment using ANARCI
Do it for heavy and light chain seperately.

In [26]:
from anarcii import Anarcii

!anarcii --seq_type antibody --mode accuracy --output seq_aligned_H2_imgt.csv seq_H.fasta
!anarcii --seq_type antibody --mode accuracy --output seq_aligned_KL2_imgt.csv seq_L.fasta

Last output saved to seq_aligned_H2_imgt.csv in scheme: None.
Last output saved to seq_aligned_KL2_imgt.csv in scheme: None.


## Combine aligned heavy chain and light chain sequences

In [27]:
H_aligned = pd.read_csv('seq_aligned_H2_imgt.csv')
L_aligned = pd.read_csv('seq_aligned_KL2_imgt.csv')

In [28]:
# https://github.com/Lailabcode/DeepSCM/blob/main/deepscm-master/seq_preprocessing.py

def seq_preprocessing():
  infile_H = pd.read_csv('seq_aligned_H2_imgt.csv')
  infile_L = pd.read_csv('seq_aligned_KL2_imgt.csv')
  outfile = open('seq_aligned_HL2.txt', "w")

  H_inclusion_list = ['1','2','3','4','5','6','7','8','9','10', \
                    '11','12','13','14','15','16','17','18','19','20', \
                    '21','22','23','24','25','26','27','28','29','30', \
                    '31','32','33','34','35','36','37','38','39','40', \
                    '41','42','43','44','45','46','47','48','49','50', \
                    '51','52','53','54','55','56','57','58','59','60', \
                    '61','62','63','64','65','66','67','68','69','70', \
                    '71','72','73','74','75','76','77','78','79','80', \
                    '81','82','83','84','85','86','87','88','89','90', \
                    '91','92','93','94','95','96','97','98','99','100', \
                    '101','102','103','104','105','106','107','108','109','110', \
                    '111','111A','111B','111C','111D','111E','111F','111G','111H', \
                    '112I','112H','112G','112F','112E','112D','112C','112B','112A','112',\
                    '113','114','115','116','117','118','119','120', \
                    '121','122','123','124','125','126','127','128']

  L_inclusion_list = ['1','2','3','4','5','6','7','8','9','10', \
                    '11','12','13','14','15','16','17','18','19','20', \
                    '21','22','23','24','25','26','27','28','29','30', \
                    '31','32','33','34','35','36','37','38','39','40', \
                    '41','42','43','44','45','46','47','48','49','50', \
                    '51','52','53','54','55','56','57','58','59','60', \
                    '61','62','63','64','65','66','67','68','69','70', \
                    '71','72','73','74','75','76','77','78','79','80', \
                    '81','82','83','84','85','86','87','88','89','90', \
                    '91','92','93','94','95','96','97','98','99','100', \
                    '101','102','103','104','105','106','107','108','109','110', \
                    '111','112','113','114','115','116','117','118','119','120', \
                    '121','122','123','124','125','126','127']

  H_dict = {'1': 0, '2':1, '3':2, '4':3, '5':4, '6':5, '7':6, '8':7, '9':8, '10':9, \
          '11':10, '12':11, '13':12, '14':13, '15':14, '16':15, '17':16, '18':17, '19':18, '20':19, \
          '21':20, '22':21, '23':22, '24':23, '25':24, '26':25, '27':26, '28':27, '29':28, '30':29, \
          '31':30, '32':31, '33':32, '34':33, '35':34, '36':35, '37':36, '38':37, '39':38, '40':39, \
          '41':40, '42':41, '43':42, '44':43, '45':44, '46':45, '47':46, '48':47, '49':48, '50':49, \
          '51':50, '52':51, '53':52, '54':53, '55':54, '56':55, '57':56, '58':57, '59':58, '60':59, \
          '61':60, '62':61, '63':62, '64':63, '65':64, '66':65, '67':66, '68':67, '69':68, '70':69, \
          '71':70, '72':71, '73':72, '74':73, '75':74, '76':75, '77':76, '78':77, '79':78, '80':79, \
          '81':80, '82':81, '83':82, '84':83, '85':84, '86':85, '87':86, '88':87, '89':88, '90':89, \
          '91':90, '92':91, '93':92, '94':93, '95':94, '96':95, '97':96, '98':97, '99':98, '100':99, \
          '101':100,'102':101,'103':102,'104':103,'105':104,'106':105,'107':106,'108':107,'109':108,'110':109, \
          '111':110,'111A':111,'111B':112,'111C':113,'111D':114,'111E':115,'111F':116,'111G':117,'111H':118, \
          '112I':119,'112H':120,'112G':121,'112F':122,'112E':123,'112D':124,'112C':125,'112B':126,'112A':127,'112':128, \
          '113':129,'114':130,'115':131,'116':132,'117':133,'118':134,'119':135,'120':136, \
          '121':137,'122':138,'123':139,'124':140,'125':141,'126':142,'127':143,'128':144}

  L_dict = {'1': 0, '2':1, '3':2, '4':3, '5':4, '6':5, '7':6, '8':7, '9':8, '10':9, \
          '11':10, '12':11, '13':12, '14':13, '15':14, '16':15, '17':16, '18':17, '19':18, '20':19, \
          '21':20, '22':21, '23':22, '24':23, '25':24, '26':25, '27':26, '28':27, '29':28, '30':29, \
          '31':30, '32':31, '33':32, '34':33, '35':34, '36':35, '37':36, '38':37, '39':38, '40':39, \
          '41':40, '42':41, '43':42, '44':43, '45':44, '46':45, '47':46, '48':47, '49':48, '50':49, \
          '51':50, '52':51, '53':52, '54':53, '55':54, '56':55, '57':56, '58':57, '59':58, '60':59, \
          '61':60, '62':61, '63':62, '64':63, '65':64, '66':65, '67':66, '68':67, '69':68, '70':69, \
          '71':70, '72':71, '73':72, '74':73, '75':74, '76':75, '77':76, '78':77, '79':78, '80':79, \
          '81':80, '82':81, '83':82, '84':83, '85':84, '86':85, '87':86, '88':87, '89':88, '90':89, \
          '91':90, '92':91, '93':92, '94':93, '95':94, '96':95, '97':96, '98':97, '99':98, '100':99, \
          '101':100,'102':101,'103':102,'104':103,'105':104,'106':105,'107':106,'108':107,'109':108,'110':109, \
          '111':110,'112':111,'113':112,'114':113,'115':114,'116':115,'117':116,'118':117,'119':118,'120':119, \
          '121':120,'122':121,'123':122,'124':123,'125':124,'126':125,'127':126,'128':127}


  N_mAbs = len(infile_H["Name"])

  for i in range(N_mAbs):
    H_tmp = 145*['-']
    L_tmp = 127*['-']
    for col in infile_H.columns:
        if(col in H_inclusion_list):
            H_tmp[H_dict[col]]=infile_H.iloc[i][col]
    for col in infile_L.columns:
        if(col in L_inclusion_list):
            L_tmp[L_dict[col]]=infile_L.iloc[i][col]

    aa_string = ''
    for aa in H_tmp+L_tmp:
         aa_string += aa
    outfile.write(infile_H.iloc[i,0]+" "+aa_string)
    outfile.write("\n")

  outfile.close()
  return

seq_preprocessing()

## DeepSP

In [30]:
def load_input_data(filename):
    name_list=[]
    seq_list=[]
    with open(filename) as datafile:
        for line in datafile:
            line = line.strip().split()
            name_list.append(line[0])
            seq_list.append(line[1])
    return name_list, seq_list

In [31]:
name_list, seq_list = load_input_data('seq_aligned_HL2.txt')
X = seq_list

In [32]:
def one_hot_encoder(s):
    d = {'A': 0, 'C': 1, 'D': 2, 'E': 3, 'F': 4, 'G': 5, 'H': 6, 'I': 7, 'K': 8, 'L': 9, 'M': 10, 'N': 11, 'P': 12, 'Q': 13, 'R': 14, 'S': 15, 'T': 16, 'V': 17, 'W': 18, 'Y': 19, '-': 20}

    x = np.zeros((len(d), len(s)))
    x[[d[c] for c in s], range(len(s))] = 1

    return x

X = [one_hot_encoder(s=x) for x in X]
X = np.transpose(np.asarray(X), (0, 2, 1))
X = np.asarray(X)

In [33]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# Add any other custom layers or classes to this dictionary if necessary
custom_objects = {
    'Sequential': Sequential,
    'Conv1D': Conv1D,
    'BatchNormalization': BatchNormalization,
    'Dropout': Dropout,
    'MaxPooling1D': MaxPooling1D,
    'Flatten': Flatten,
    'Dense': Dense
}

# Function to load and compile a model
def load_and_compile_model(json_path, weights_path):
    # Load model architecture from JSON file
    with open(json_path, 'r') as json_file:
        loaded_model_json = json_file.read()

    # Load model from JSON
    loaded_model = model_from_json(loaded_model_json, custom_objects=custom_objects)

    # Load weights into model
    loaded_model.load_weights(weights_path)

    # Compile the model with optimizer and loss function
    loaded_model.compile(optimizer='adam', loss='mae', metrics=['mae'])

    return loaded_model

# Assuming X is already defined with your input data

# Load and predict for sap_pos
sap_pos_model = load_and_compile_model('Conv1D_regressionSAPpos.json', 'Conv1D_regression_SAPpos.h5')
sap_pos = sap_pos_model.predict(X)

# Load and predict for scm_neg
scm_neg_model = load_and_compile_model('Conv1D_regressionSCMneg.json', 'Conv1D_regression_SCMneg.h5')
scm_neg = scm_neg_model.predict(X)

# Load and predict for scm_pos
scm_pos_model = load_and_compile_model('Conv1D_regressionSCMpos.json', 'Conv1D_regression_SCMpos.h5')
scm_pos = scm_pos_model.predict(X)

# Now sap_pos, scm_neg, and scm_pos contain the predictions for each respective model


5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 81ms/step
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
4/5 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/stepWARNING:tensorflow:5 out of the last 11 calls to <function TensorFlowTrainer.make_predict_function.<locals>.one_step_on_data_distributed at 0x7fdf26c53740> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 95ms/step


In [34]:
features = ['Name', 'SAP_pos_CDRH1','SAP_pos_CDRH2','SAP_pos_CDRH3','SAP_pos_CDRL1','SAP_pos_CDRL2','SAP_pos_CDRL3','SAP_pos_CDR','SAP_pos_Hv','SAP_pos_Lv','SAP_pos_Fv',
          'SCM_neg_CDRH1','SCM_neg_CDRH2','SCM_neg_CDRH3','SCM_neg_CDRL1','SCM_neg_CDRL2','SCM_neg_CDRL3','SCM_neg_CDR','SCM_neg_Hv','SCM_neg_Lv','SCM_neg_Fv',
          'SCM_pos_CDRH1','SCM_pos_CDRH2','SCM_pos_CDRH3','SCM_pos_CDRL1','SCM_pos_CDRL2','SCM_pos_CDRL3','SCM_pos_CDR','SCM_pos_Hv','SCM_pos_Lv','SCM_pos_Fv']
df = pd.concat([pd.DataFrame(name_list), pd.DataFrame(sap_pos),  pd.DataFrame(scm_neg), pd.DataFrame(scm_pos)], ignore_index=True, axis=1,); df.columns = features
df.to_csv('DeepSP_descriptors_anarci2_Abdev.csv', index=False)
df

,Name,SAP_pos_CDRH1,SAP_pos_CDRH2,SAP_pos_CDRH3,SAP_pos_CDRL1,SAP_pos_CDRL2,SAP_pos_CDRL3,SAP_pos_CDR,SAP_pos_Hv,SAP_pos_Lv,...,SCM_pos_CDRH1,SCM_pos_CDRH2,SCM_pos_CDRH3,SCM_pos_CDRL1,SCM_pos_CDRL2,SCM_pos_CDRL3,SCM_pos_CDR,SCM_pos_Hv,SCM_pos_Lv,SCM_pos_Fv
0,abituzumab,3.859764,4.345273,8.718716,2.148599,4.696163,6.794335,30.680004,45.527824,36.425133,...,33.356182,50.214882,58.551708,37.382198,99.530861,19.154770,301.265564,877.353760,1176.290283,2044.643311
1,abrilumab,3.417136,1.079907,1.828652,2.087438,2.402200,6.014892,17.925526,43.011524,31.231604,...,3.424593,-1.281652,7.433341,36.730270,8.360390,19.308479,67.659920,919.620972,941.504150,1849.138794
2,adalimumab,2.134783,2.524245,14.445071,1.904740,3.589351,3.172182,27.496794,58.417648,30.517651,...,3.183182,19.583357,29.513483,116.769501,41.759361,55.548267,263.838928,907.113037,1219.444458,2109.088623
3,alemtuzumab,2.272270,3.696255,5.040778,2.220291,2.762959,3.326095,20.664444,52.663429,32.287445,...,31.126842,109.974251,113.784378,59.374569,59.348289,156.546021,539.401306,1461.241699,1302.770508,2748.457520
4,alirocumab,2.335272,0.485208,5.749701,6.340402,2.478856,4.473296,23.008087,52.346256,43.460670,...,87.390930,30.730070,22.045889,163.355850,64.857002,19.051237,393.546326,1270.465210,966.631592,2228.184570
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132,vedolizumab,2.809752,0.265223,11.760819,5.934891,2.963002,2.818470,27.306589,51.973351,51.453320,...,30.365030,-3.822329,4.026950,147.944061,50.612793,64.961990,289.120789,982.951416,1155.896118,2126.845215
133,veltuzumab,2.509693,4.151934,14.463856,3.159272,4.066860,2.541194,31.233425,43.669392,37.644169,...,34.895672,0.024304,39.568436,32.313793,35.552647,30.637224,166.967972,1078.540283,1059.886108,2115.083008
134,visilizumab,6.160062,3.600580,15.227443,2.185714,3.861778,2.654676,34.395943,57.106976,32.811314,...,100.690536,106.182350,75.532433,-2.440957,69.549660,18.262016,372.927856,1269.878418,1078.142334,2312.872559
135,zalutumumab,1.994143,5.136198,18.606033,0.970755,2.458154,5.581917,34.689407,63.186798,32.754875,...,29.383791,5.332729,43.206482,1.038679,2.665065,29.451338,108.277626,1160.793091,941.983887,2081.374512
